In [1]:
import sys
from sqlalchemy import String, DateTime
from sqlalchemy.dialects.mssql import NVARCHAR
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import uuid
from datetime import datetime, timezone

from src.config import validate_config

from src.quickbooks import (
    get_accounts,
    get_customers,
    get_journal_entries
)

from src.azure_sql import (
    get_engine,
    test_connection,
    write_dataframe,
    read_sql
)

from src.transformations import build_raw_dataframe

In [3]:
validate_config()

engine = get_engine()
test_connection(engine)

batch_id = str(uuid.uuid4())
extracted_at = datetime.now(timezone.utc)

print("Batch ID:", batch_id)
print("Extracted at:", extracted_at)

Batch ID: 09f210a6-6c87-4714-87e9-c8ffd1bf6a23
Extracted at: 2026-09-15 22:17:05.356756+00:00


In [4]:
accounts = get_accounts()
customers = get_customers()

journal_entries = get_journal_entries(
    start_date="2023-09-01",
    end_date="2026-08-31"
)

print("Accounts:", len(accounts))
print("Customers:", len(customers))
print("Journal Entries:", len(journal_entries))

Accounts: 89
Customers: 411
Journal Entries: 39


In [5]:
df_accounts_raw = build_raw_dataframe(
    accounts,
    batch_id=batch_id,
    extracted_at=extracted_at
)

df_customers_raw = build_raw_dataframe(
    customers,
    batch_id=batch_id,
    extracted_at=extracted_at
)

df_journals_raw = build_raw_dataframe(
    journal_entries,
    batch_id=batch_id,
    extracted_at=extracted_at
)

In [6]:
display(df_accounts_raw.head())
display(df_customers_raw.head())
display(df_journals_raw.head())

,entity_id,batch_id,extracted_at,payload_json
0,69,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Name"": ""Accounting"", ""SubAccount"": true, ""Pa..."
1,33,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Name"": ""Accounts Payable (A/P)"", ""SubAccount..."
2,84,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Name"": ""Accounts Receivable (A/R)"", ""SubAcco..."
3,7,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Name"": ""Advertising"", ""SubAccount"": false, ""..."
4,89,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Name"": ""Arizona Dept. of Revenue Payable"", ""..."


,entity_id,batch_id,extracted_at,payload_json
0,1,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Taxable"": true, ""BillAddr"": {""Id"": ""2"", ""Lin..."
1,411,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Taxable"": false, ""BillAddr"": {""Id"": ""449"", ""..."
2,254,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Taxable"": false, ""BillAddr"": {""Id"": ""292"", ""..."
3,287,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Taxable"": false, ""BillAddr"": {""Id"": ""325"", ""..."
4,167,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Taxable"": false, ""BillAddr"": {""Id"": ""205"", ""..."


,entity_id,batch_id,extracted_at,payload_json
0,180,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Adjustment"": false, ""TotalAmt"": 0, ""domain"":..."
1,179,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Adjustment"": false, ""TotalAmt"": 0, ""domain"":..."
2,178,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Adjustment"": false, ""TotalAmt"": 0, ""domain"":..."
3,177,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Adjustment"": false, ""TotalAmt"": 0, ""domain"":..."
4,176,09f210a6-6c87-4714-87e9-c8ffd1bf6a23,2026-09-15 22:17:05.356756+00:00,"{""Adjustment"": false, ""TotalAmt"": 0, ""domain"":..."


In [7]:
bronze_dtype = {
    "entity_id": String(100),
    "batch_id": String(36),
    "extracted_at": DateTime(timezone=True),
    "payload_json": NVARCHAR(None)  # NVARCHAR(MAX)
}

In [8]:
write_dataframe(
    df_accounts_raw,
    table="qbo_accounts_raw",
    schema="bronze",
    if_exists="append",
    engine=engine,
    dtype=bronze_dtype
)

write_dataframe(
    df_customers_raw,
    table="qbo_customers_raw",
    schema="bronze",
    if_exists="append",
    engine=engine,
    dtype=bronze_dtype
)

write_dataframe(
    df_journals_raw,
    table="qbo_journal_entries_raw",
    schema="bronze",
    if_exists="append",
    engine=engine,
    dtype=bronze_dtype
)

In [9]:
check = read_sql(
    f"""
    SELECT
        'accounts' AS entity,
        COUNT(*) AS rows
    FROM bronze.qbo_accounts_raw
    WHERE batch_id = '{batch_id}'

    UNION ALL

    SELECT
        'customers',
        COUNT(*)
    FROM bronze.qbo_customers_raw
    WHERE batch_id = '{batch_id}'

    UNION ALL

    SELECT
        'journal_entries',
        COUNT(*)
    FROM bronze.qbo_journal_entries_raw
    WHERE batch_id = '{batch_id}'
    """,
    engine
)

display(check)

,entity,rows
0,accounts,89
1,customers,411
2,journal_entries,39
